<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

I reviewed two findings from *The State of AI-Driven SEO, March 2026*. These questions are intended to clarify what the reported evidence supports and how the comparisons could be made more informative. They are not claims that the analysis is incorrect.

### Finding 1 — The Anatomy of Growing Content (paper, page 6)

**Reported finding:** The paper compares 74,187 pages with rising impressions and 45,272 pages with falling impressions. The rising group averaged 3,180 words and 184 days of age, compared with 2,311 words and 230 days for the falling group.

**Where does the label come from?** The paper defines trend direction using the change in impressions over the latest 30 days compared with the previous 30 days. Its metric definitions classify growth above 10% as "Up" and decline greater than 10% as "Down" (page 5). These groups are then compared on content characteristics such as word count and age.

**My methodology question:** Could the analysis clarify whether the word-count and age differences remain similar after comparing pages within the same content type, age range, or prior-visibility range? Older pages may differ from younger pages in several ways besides their word count. Because the comparison is observational, I would interpret the results as an association between content characteristics and observed impression direction, rather than evidence that adding words causes growth.

### Finding 2 — The Freshness Multiplier (paper, page 9)

**Reported finding:** The paper reports a health-score comparison of 34.5 versus 10.7 and an impression comparison of 4,039 versus 71 for mature content associated with a recent refresh. These figures are presented as a 3.2× health difference and a 57× impression difference.

**Where does the label come from?** This finding does not use a single binary prediction label. It groups mature content by refresh recency and compares observed health scores and impressions. The health score is a composite metric that includes impressions, position, CTR, and scroll depth, as defined on page 5.

**My methodology question:** How were the refreshed and comparison groups selected, and were their measurement windows and starting performance comparable? Pages chosen for refreshing may already differ in quality, visibility, or business priority. I would also distinguish the health-score difference from the impression difference because impressions already contribute to the health score. Comparing similar refreshed and unrefreshed pages over aligned periods would help assess how robust the observed gap is.

### What I take from this review

Both findings provide useful portfolio-level evidence for deciding what to investigate. The methodology questions concern the interpretation of the comparisons, not whether the reported numbers should be dismissed. I would keep the conclusions observational unless a suitable validation design supports stronger claims.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1: Evidence check for the two research-paper findings.
# This cell documents the reported comparisons and the questions
# they raise. It does not claim to reproduce the paper's analysis.

import pandas as pd
from IPython.display import display

paper_audit = pd.DataFrame([
    {
        "finding": "The Anatomy of Growing Content",
        "paper_page": 6,
        "comparison": "Rising vs. falling impression pages",
        "reported_group_sizes": "74,187 vs. 45,272",
        "reported_metrics": (
            "Mean word count: 3,180 vs. 2,311; "
            "mean age: 184 vs. 230 days"
        ),
        "methodology_question": (
            "Do the differences persist after accounting for "
            "content type, age, and prior visibility?"
        ),
        "interpretation_limit": (
            "Observational association; does not establish "
            "that longer content causes impression growth."
        ),
    },
    {
        "finding": "The Freshness Multiplier",
        "paper_page": 9,
        "comparison": "Mature content associated with recent refresh",
        "reported_group_sizes": "Not recorded in this audit cell",
        "reported_metrics": (
            "Health score: 34.5 vs. 10.7; "
            "impressions: 4,039 vs. 71"
        ),
        "methodology_question": (
            "Were groups comparable at baseline and measured "
            "over aligned windows?"
        ),
        "interpretation_limit": (
            "The observed gap does not establish a causal "
            "refresh effect; health score includes impressions."
        ),
    },
])

print("RESEARCH-PAPER METHODOLOGY AUDIT")
print("================================")
display(paper_audit)

# Basic documentation checks.
assert len(paper_audit) == 2
assert paper_audit["finding"].is_unique
assert paper_audit["methodology_question"].notna().all()
assert paper_audit["interpretation_limit"].notna().all()

print("\nEvidence-check assertions passed.")
print(
    "Note: reported figures are transcribed from the paper "
    "summary in Section 1; this cell does not independently "
    "recalculate the paper's results."
)


RESEARCH-PAPER METHODOLOGY AUDIT


,finding,paper_page,comparison,reported_group_sizes,reported_metrics,methodology_question,interpretation_limit
0,The Anatomy of Growing Content,6,Rising vs. falling impression pages,"74,187 vs. 45,272","Mean word count: 3,180 vs. 2,311; mean age: 18...",Do the differences persist after accounting fo...,Observational association; does not establish ...
1,The Freshness Multiplier,9,Mature content associated with recent refresh,Not recorded in this audit cell,"Health score: 34.5 vs. 10.7; impressions: 4,03...",Were groups comparable at baseline and measure...,The observed gap does not establish a causal r...



Evidence-check assertions passed.
Note: reported figures are transcribed from the paper summary in Section 1; this cell does not independently recalculate the paper's results.



## 2. My model under an honest split (before/after)

### Objective

Compare model evaluation under two split strategies to understand how the evaluation changes when pages from the same client are kept together.

### Before: Random row-level split

A random row-level split can place pages from the same client in both the training and test sets. Because pages belonging to one client may share characteristics, the model may benefit from client-specific patterns that are also present in the test set.

This result is useful as a diagnostic comparison, but it may overstate performance on entirely unseen clients.

### After: Client-grouped split

Use `GroupShuffleSplit` with `client_hash_id` as the grouping variable. All rows belonging to a client must remain in either the training set or the test set, never both.

The Week 5 grouped evaluation used a 75/25 split with `random_state=42`. It placed 35 clients in training and 12 in testing, with zero client overlap.

### Evaluation protocol

For both split strategies:

- Use the same feature columns and target definition.
- Fit preprocessing steps using only the training data.
- Train the same Logistic Regression model with the same settings.
- Reconstruct and evaluate the same baseline score.
- Report the test-set decline rate, Precision@20, Precision@50, ROC-AUC, and average precision.
- Report client overlap to verify whether the split isolates clients.

### Interpretation

Compare the metrics from both experiments rather than assuming that either split must perform better. A difference in performance may indicate that the split strategy affects the evaluation.

**Important limitation:** A client-grouped split tests generalization to unseen clients; it does not establish performance on future calendar periods. A separate time-based evaluation would be needed to assess that question.


In [3]:

# Section 2: Compare random row-level and client-grouped evaluation.
# This cell builds its own feature frame; it does not depend on Week 5 variables.

%pip -q install duckdb huggingface_hub scikit-learn pandas

import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Authenticate with Hugging Face using the existing Colab secret.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it in Colab Secrets and enable "
        "notebook access before running this cell."
    )

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN],
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet('{REL}/fact_content_daily_performance/"
    "month=2026-03/*.parquet')"
)
APRIL = (
    f"read_parquet('{REL}/fact_content_daily_performance/"
    "month=2026-04/*.parquet')"
)

# Build March features and April target separately.
feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_mean_position,
        COUNT(DISTINCT CASE
            WHEN gsc_impressions > 0 THEN report_date
        END) AS march_impression_days,
        DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1
            AS march_active_days_span
    FROM {MARCH}
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {APRIL}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_mean_position,
    m.march_impression_days,
    m.march_active_days_span,
    a.april_impressions,
    100.0 * (a.april_impressions - m.march_impressions)
        / NULLIF(m.march_impressions, 0) AS april_trend_pct
FROM march m
JOIN april a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id
WHERE m.march_impressions > 0
"""

feature_frame = con.sql(feature_sql).df()
con.close()

FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_mean_position",
    "march_impression_days",
    "march_active_days_span",
]

feature_frame["april_decline_label"] = (
    feature_frame["april_trend_pct"] < -20.0
).astype(int)

X = feature_frame[FEATURES].copy()
y = feature_frame["april_decline_label"].astype(int)
groups = feature_frame["client_hash_id"]

print("FEATURE FRAME")
print("=============")
print(f"Content items: {len(feature_frame):,}")
print(f"Distinct clients: {groups.nunique():,}")
print(f"April decline rate: {y.mean():.1%}")

# BEFORE: random row-level split.
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(feature_frame)),
    test_size=0.25,
    random_state=42,
    stratify=y,
)

# AFTER: client-grouped split.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)
group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

splits = {
    "Random row-level": (random_train_idx, random_test_idx),
    "Client-grouped": (group_train_idx, group_test_idx),
}

def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_array))
    top_idx = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_array[top_idx].mean())

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

split_checks = []
metric_rows = []

for split_name, (train_idx, test_idx) in splits.items():
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])
    overlap = train_clients & test_clients

    split_checks.append({
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_clients": len(train_clients),
        "test_clients": len(test_clients),
        "client_overlap": len(overlap),
        "train_decline_rate": y_train.mean(),
        "test_decline_rate": y_test.mean(),
    })

    model = make_model()
    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_test)[:, 1]

    # Week-4 rule baseline uses March features only.
    impressions = X_test["march_impressions"]
    clicks = X_test["march_clicks"]
    position = X_test["march_mean_position"]

    ctr = (
        100 * clicks / impressions.replace(0, np.nan)
    ).fillna(0)

    ctr_gap = np.clip(0.5 - ctr, 0, 0.5) / 0.5
    visibility_gate = (
        (impressions >= 500)
        & (position > 0)
        & (position <= 20)
    ).astype(int)

    baseline_scores = (
        np.log1p(impressions) * ctr_gap * visibility_gate
    ).to_numpy()

    for method, scores in [
        ("Week-4 rule baseline", baseline_scores),
        ("Logistic Regression", model_scores),
    ]:
        metric_rows.append({
            "split": split_name,
            "method": method,
            "test_decline_rate": y_test.mean(),
            "precision_at_20": precision_at_k(y_test, scores, 20),
            "precision_at_50": precision_at_k(y_test, scores, 50),
            "roc_auc": roc_auc_score(y_test, scores),
            "average_precision": average_precision_score(y_test, scores),
        })

print("\nSPLIT CHECKS")
print("============")
display(pd.DataFrame(split_checks).round(4))

print("\nMODEL AND BASELINE COMPARISON")
print("============================")
display(pd.DataFrame(metric_rows).round(4))

# A grouped split must never share clients between training and test.
grouped_overlap = next(
    row["client_overlap"]
    for row in split_checks
    if row["split"] == "Client-grouped"
)
assert grouped_overlap == 0, "Client overlap found in grouped split."

print("\nValidation passed: client-grouped split has zero client overlap.")
print(
    "Note: the two strategies use different test rows. "
    "Metric differences may reflect both split design and test-set composition."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FEATURE FRAME
Content items: 176,737
Distinct clients: 47
April decline rate: 53.2%

SPLIT CHECKS


,split,train_rows,test_rows,train_clients,test_clients,client_overlap,train_decline_rate,test_decline_rate
0,Random row-level,132552,44185,47,45,45,0.5319,0.5319
1,Client-grouped,133473,43264,35,12,0,0.5575,0.4529



MODEL AND BASELINE COMPARISON


,split,method,test_decline_rate,precision_at_20,precision_at_50,roc_auc,average_precision
0,Random row-level,Week-4 rule baseline,0.5319,0.70,0.60,0.5128,0.5507
1,Random row-level,Logistic Regression,0.5319,0.55,0.64,0.5947,0.5890
2,Client-grouped,Week-4 rule baseline,0.4529,0.65,0.56,0.5178,0.4750
3,Client-grouped,Logistic Regression,0.4529,0.65,0.54,0.5779,0.5150



Validation passed: client-grouped split has zero client overlap.
Note: the two strategies use different test rows. Metric differences may reflect both split design and test-set composition.



## 3. Leakage audit

### Target leakage

The target, `april_decline_label`, is derived from April impressions relative to March impressions. The feature matrix contains only the five contracted March features. April impressions and `april_trend_pct` are used to construct the target and are not included as model features.

### Identifier leakage

`client_hash_id` and `content_hash_id` are used to join records and define groups, but they are excluded from the feature matrix. The client-grouped split confirms that no client appears in both the training and test sets.

### Preprocessing leakage

Median imputation and standardization are fitted within a scikit-learn pipeline on the training data only. The fitted transformations are then applied to the test data.

### Remaining limitations

The random row-level split has client overlap and is not the preferred estimate of generalization to unseen clients. The client-grouped split addresses this issue, but it is not a time-based evaluation. The model has not yet demonstrated performance on a later, unseen calendar period.

**Conclusion:** The main feature, target, grouping, and preprocessing checks pass for the client-grouped experiment. This does not prove that every possible source of leakage has been eliminated.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3 — Leakage audit

# 1. Confirm that only March features are used for model input.
forbidden_features = {
    "april_decline_label",
    "april_impressions",
    "april_trend_pct",
    "client_hash_id",
    "content_hash_id",
}

assert set(FEATURES).isdisjoint(forbidden_features), (
    f"Leakage risk: forbidden columns found in FEATURES: "
    f"{set(FEATURES) & forbidden_features}"
)

assert all(feature in feature_frame.columns for feature in FEATURES), (
    "At least one configured feature is missing from feature_frame."
)

assert "april_decline_label" in feature_frame.columns
assert "client_hash_id" in feature_frame.columns

# 2. Confirm that the target is derived from April trend, not used as a feature.
expected_target = (feature_frame["april_trend_pct"] < -20).astype(int)

target_match = (
    feature_frame["april_decline_label"].astype(int)
    == expected_target
).all()

assert target_match, "Target labels do not match the documented April trend rule."

# 3. Confirm the grouped split keeps clients separate.
train_clients = set(
    feature_frame.loc[train_idx, "client_hash_id"].dropna()
)
test_clients = set(
    feature_frame.loc[test_idx, "client_hash_id"].dropna()
)

client_overlap = train_clients & test_clients
assert not client_overlap, (
    f"Client leakage in grouped split: {len(client_overlap)} clients overlap."
)

# 4. Confirm the grouped split has no row-index overlap.
assert len(set(train_idx) & set(test_idx)) == 0, (
    "Train and test row indices overlap."
)

# 5. Report checks and remaining limitations.
print("Leakage audit passed:")
print(f"- Model features: {len(FEATURES)} configured March features")
print("- April target columns and identifiers excluded from FEATURES")
print("- Target matches the April trend < -20% rule")
print("- No client overlap in the grouped split")
print("- No train/test row-index overlap")
print()
print("Limitations:")
print("- The random row-level split still shares clients across train and test.")
print("- A client-grouped split evaluates unseen clients, not future time periods.")
print("- These checks do not prove that every possible leakage source is absent.")


Leakage audit passed:
- Model features: 5 configured March features
- April target columns and identifiers excluded from FEATURES
- Target matches the April trend < -20% rule
- No client overlap in the grouped split
- No train/test row-index overlap

Limitations:
- The random row-level split still shares clients across train and test.
- A client-grouped split evaluates unseen clients, not future time periods.
- These checks do not prove that every possible leakage source is absent.



## 4. Rewrite the claim

**Original claim:** The model can reliably predict which content pages will lose impressions.

**Evidence-based rewrite:** In this experiment, a logistic regression model using five March features performed better than the baseline on ROC-AUC and average precision under both random row-level and client-grouped splits. Under the client-grouped split, it matched the baseline at Precision@20 and performed slightly worse at Precision@50. These results provide limited evidence of predictive signal for identifying pages that subsequently declined in impressions.

**What the evidence does not establish:** The results do not demonstrate reliable performance on future calendar periods, prove that the model generalizes to every unseen client, or establish that the selected features cause impression declines. The client-grouped evaluation is more appropriate for assessing performance on unseen clients, but further validation on a later time period is needed before making a stronger reliability claim.


In [6]:

# Section 4 — Claim rewrite: evidence checks

# Metrics reported by Cell 4
evidence = {
    "Random split": {
        "Baseline": {
            "Precision@20": 0.70,
            "Precision@50": 0.60,
            "ROC-AUC": 0.5128,
            "Average Precision": 0.5507,
        },
        "Logistic Regression": {
            "Precision@20": 0.55,
            "Precision@50": 0.64,
            "ROC-AUC": 0.5947,
            "Average Precision": 0.5890,
        },
    },
    "Client-grouped split": {
        "Baseline": {
            "Precision@20": 0.65,
            "Precision@50": 0.56,
            "ROC-AUC": 0.5178,
            "Average Precision": 0.4750,
        },
        "Logistic Regression": {
            "Precision@20": 0.65,
            "Precision@50": 0.54,
            "ROC-AUC": 0.5779,
            "Average Precision": 0.5150,
        },
    },
}

for split_name, models in evidence.items():
    print(f"\n{split_name}")
    print("-" * len(split_name))

    for model_name, metrics in models.items():
        print(f"\n{model_name}")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")

print("\nClaim review")
print("-" * 50)
print("Logistic regression improves ROC-AUC and average precision")
print("over the baseline under both split strategies.")
print("For the client-grouped split, Precision@20 ties the baseline,")
print("while Precision@50 is slightly lower.")
print("These results do not establish future-period reliability")
print("or demonstrate that the features cause impression declines.")



Random split
------------

Baseline
  Precision@20: 0.7000
  Precision@50: 0.6000
  ROC-AUC: 0.5128
  Average Precision: 0.5507

Logistic Regression
  Precision@20: 0.5500
  Precision@50: 0.6400
  ROC-AUC: 0.5947
  Average Precision: 0.5890

Client-grouped split
--------------------

Baseline
  Precision@20: 0.6500
  Precision@50: 0.5600
  ROC-AUC: 0.5178
  Average Precision: 0.4750

Logistic Regression
  Precision@20: 0.6500
  Precision@50: 0.5400
  ROC-AUC: 0.5779
  Average Precision: 0.5150

Claim review
--------------------------------------------------
Logistic regression improves ROC-AUC and average precision
over the baseline under both split strategies.
For the client-grouped split, Precision@20 ties the baseline,
while Precision@50 is slightly lower.
These results do not establish future-period reliability
or demonstrate that the features cause impression declines.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.